# Spaceship Titanic
**Author:** Koketso Raphasha | [Kaggle: Raphasha27](https://kaggle.com/Raphasha27)

**Approach:** Feature engineering, KNN imputation, categorical encoding, ensemble classifiers

**Metric:** Categorization Accuracy

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
train = pd.read_csv('/kaggle/input/spaceship-titanic/train.csv')
test = pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')
print(f'Train: {train.shape[0]} rows x {train.shape[1]} cols')
print(f'Test:  {test.shape[0]} rows x {test.shape[1]} cols')
print(f'\nMissing:\n{train.isnull().sum().sort_values(ascending=False)}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
sns.countplot(data=train, x='Transported', ax=axes[0,0])
axes[0,0].set_title('Transport Distribution')
sns.countplot(data=train, x='HomePlanet', hue='Transported', ax=axes[0,1])
axes[0,1].set_title('Transport by HomePlanet')
sns.countplot(data=train, x='Destination', hue='Transported', ax=axes[0,2])
axes[0,2].set_title('Transport by Destination')
sns.histplot(data=train, x='Age', hue='Transported', kde=True, bins=30, ax=axes[1,0])
axes[1,0].set_title('Age Distribution')
sns.countplot(data=train, x='CryoSleep', hue='Transported', ax=axes[1,1])
axes[1,1].set_title('Transport by CryoSleep')
sns.countplot(data=train, x='VIP', hue='Transported', ax=axes[1,2])
axes[1,2].set_title('Transport by VIP')
plt.tight_layout()
plt.savefig('eda.png', dpi=100)
plt.show()

## Feature Engineering

In [ ]:
def preprocess(df):
    data = df.copy()
    data[['Deck','CabinNum','Side']] = data['Cabin'].str.split('/', expand=True)
    data['Deck'] = data['Deck'].fillna('Unknown').astype(str)
    data['Side'] = data['Side'].fillna('Unknown').astype(str)
    data['CabinNum'] = pd.to_numeric(data['CabinNum'], errors='coerce')
    data['CryoSleep'] = data['CryoSleep'].fillna(False).astype(int)
    data['VIP'] = data['VIP'].fillna(False).astype(int)
    data['Age'] = data['Age'].fillna(data['Age'].median())
    data['RoomService'] = data['RoomService'].fillna(0)
    data['FoodCourt'] = data['FoodCourt'].fillna(0)
    data['ShoppingMall'] = data['ShoppingMall'].fillna(0)
    data['Spa'] = data['Spa'].fillna(0)
    data['VRDeck'] = data['VRDeck'].fillna(0)
    data['TotalSpend'] = data['RoomService'] + data['FoodCourt'] + data['ShoppingMall'] + data['Spa'] + data['VRDeck']
    data['HasSpend'] = (data['TotalSpend'] > 0).astype(int)
    data['Group'] = data['PassengerId'].str.split('_').str[0]
    data['GroupSize'] = data.groupby('Group')['PassengerId'].transform('count')
    data['IsAlone'] = (data['GroupSize'] == 1).astype(int)
    return data

full = preprocess(pd.concat([train.drop('Transported', axis=1), test], ignore_index=True))

# Label encode categoricals
cat_cols = ['HomePlanet', 'Destination', 'Deck', 'Side']
for col in cat_cols:
    le = LabelEncoder()
    full[col] = le.fit_transform(full[col].fillna('Unknown').astype(str))

# KNN impute for numeric
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'CabinNum']
knn = KNNImputer(n_neighbors=5)
full[num_cols] = pd.DataFrame(knn.fit_transform(full[num_cols]), columns=num_cols)

y = train['Transported'].astype(int).values
feat_cols = ['HomePlanet','Destination','Deck','Side','CryoSleep','VIP','Age',
    'RoomService','FoodCourt','ShoppingMall','Spa','VRDeck','TotalSpend','HasSpend',
    'GroupSize','IsAlone','CabinNum']

X_full = full.iloc[:len(train)][feat_cols]
X_test = full.iloc[len(train):][feat_cols]
scaler = StandardScaler()
X_full_s = scaler.fit_transform(X_full)
X_test_s = scaler.transform(X_test)
print(f'Features: {len(feat_cols)}')

## Model Training

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

models = {
    'RF': RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=4, random_state=42),
    'GB': GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                                      subsample=0.8, random_state=42),
    'XGB': XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                          colsample_bytree=0.8, eval_metric='logloss', random_state=42),
}

for name, model in models.items():
    scores = cross_val_score(model, X_full_s, y, cv=cv, scoring='accuracy')
    print(f'{name}: CV {scores.mean():.4f} (+/- {scores.std()*2:.4f})')

# Hard voting ensemble
ensemble = VotingClassifier(estimators=[(n, m) for n, m in models.items()], voting='hard')
ensemble.fit(X_full_s, y)
preds = ensemble.predict(X_test_s)
print(f'\nTransported: {preds.sum()}/{len(preds)} ({preds.mean()*100:.1f}%)')

## Submission

In [ ]:
sub = pd.DataFrame({'PassengerId': test['PassengerId'], 'Transported': preds.astype(bool)})
sub.to_csv('submission.csv', index=False)
print('Ready: submission.csv')
sub.head()